# Ex.No 9 — Simple Code Optimization Techniques (Constant Folding, Strength Reduction, Algebraic Transformation)


## AIM
To write a program using FLEX and BISON to implement simple code optimization techniques such as constant folding, strength reduction and algebraic simplification, applied while parsing three-address code style assignment statements.


## ALGORITHM / PROCEDURE
1. Use FLEX to tokenize the input statements into identifiers, numbers and operators, passing them to BISON.
2. In BISON, define a grammar for assignment statements of the form `id = expr ;`.
3. While reducing an `expr` production:
   - **Constant Folding**: if both operands are numeric constants, evaluate the operation immediately and replace it with the result.
   - **Algebraic Simplification**: apply rules such as `x + 0 -> x`, `x - 0 -> x`, `x * 1 -> x`, `x / 1 -> x`.
   - **Strength Reduction**: replace `x * 2` with `x + x`.
4. Print the optimized right-hand side once the statement is fully reduced.
5. Repeat for every statement until the input ends.

**Procedure**
1. Create `optimize.l` to tokenize identifiers, numbers, and operators `+ - * / = ;`.
2. Create `optimize.y` with a grammar for assignment statements and expressions; in the semantic actions for each `expr` rule, check whether constant folding, strength reduction, or algebraic simplification applies, and print a comment indicating which optimization fired.
3. Compile: `flex optimize.l` → `bison -d optimize.y` → `gcc lex.yy.c optimize.tab.c -o optimize -lfl`.
4. Run `./optimize`, enter three-address-code-style statements ending with `;` (terminate with Ctrl+D), and verify constants are folded, `x*1`/`x/1`/`x+0`/`x-0` are simplified, and `x*2` is converted to `x+x`.


## PSEUDOCODE / LOGIC
```
GRAMMAR:
    stmt -> ID '=' expr ';'                { PRINT ID "=" expr }
    expr -> NUM | ID
          | expr '+' expr | expr '-' expr | expr '*' expr | expr '/' expr

ON reducing (left OP right):
    IF left and right are both numeric constants THEN
        result = evaluate(left OP right)                 // Constant Folding
    ELSE IF OP == '+' and right == "0" THEN result = left // Algebraic Simplification
    ELSE IF OP == '+' and left  == "0" THEN result = right
    ELSE IF OP == '-' and right == "0" THEN result = left
    ELSE IF OP == '*' and right == "1" THEN result = left
    ELSE IF OP == '*' and right == "2" THEN result = left + " + " + left   // Strength Reduction
    ELSE IF OP == '/' and right == "1" THEN result = left
    ELSE result = left + OP + right                       // no optimization applies

END
```


## PROGRAM & OUTPUT
The cells below contain the source program (FLEX/BISON/C) and its executed output.


In [83]:
# ============================================================
# SIMPLE CODE OPTIMIZATION USING FLEX AND BISON
# Constant Folding, Strength Reduction,
# Algebraic Transformation
# Google Colab - Single Cell
# ============================================================

# Install FLEX, BISON and GCC
!apt-get update -qq
!apt-get install -y flex bison gcc -qq


# ============================================================
# Create optimize.l
# ============================================================

with open("optimize.l", "w") as f:
    f.write(r'''
%{
#include "optimize.tab.h"
#include <string.h>
#include <stdlib.h>
%}

%option noyywrap

%%

[a-zA-Z][a-zA-Z0-9]* {
    yylval.str = strdup(yytext);
    return ID;
}

[0-9]+ {
    yylval.str = strdup(yytext);
    return NUM;
}

"="     { return '='; }
"+"     { return '+'; }
"-"     { return '-'; }
"*"     { return '*'; }
"/"     { return '/'; }
";"     { return ';'; }

[ \t\n]+ {
    /* Ignore whitespace */
}

. {
    return yytext[0];
}

%%
''')


# ============================================================
# Create optimize.y
# ============================================================

with open("optimize.y", "w") as f:
    f.write(r'''
%{
#include <stdio.h>
#include <string.h>
#include <stdlib.h>
#include <ctype.h>

int yylex(void);
int yyerror(char *s);

/* Check whether a string is a number */
int isNumber(char *s)
{
    int i;

    if (s == NULL || s[0] == '\0')
        return 0;

    for (i = 0; s[i] != '\0'; i++)
    {
        if (!isdigit((unsigned char)s[i]))
            return 0;
    }

    return 1;
}
%}

%union {
    char *str;
}

%token <str> ID NUM
%type <str> expr

%left '+' '-'
%left '*' '/'

%%

stmt_list:
      stmt_list stmt
    | stmt
    ;

stmt:
    ID '=' expr ';'
    {
        printf("%s = %s\n", $1, $3);
    }
    ;

expr:
      NUM
      {
          $$ = $1;
      }

    | ID
      {
          $$ = $1;
      }

    | expr '+' expr
      {
          char buf[100];

          /* Constant Folding */
          if (isNumber($1) && isNumber($3))
          {
              sprintf(buf, "%d", atoi($1) + atoi($3));
              $$ = strdup(buf);

              printf("// Constant Folding: %s + %s -> %s\n",
                     $1, $3, $$);
          }

          /* Algebraic Transformation: x + 0 -> x */
          else if (strcmp($3, "0") == 0)
          {
              $$ = strdup($1);

              printf("// Algebraic Simplification: x + 0 -> x\n");
          }

          /* Algebraic Transformation: 0 + x -> x */
          else if (strcmp($1, "0") == 0)
          {
              $$ = strdup($3);

              printf("// Algebraic Simplification: 0 + x -> x\n");
          }

          else
          {
              sprintf(buf, "%s + %s", $1, $3);
              $$ = strdup(buf);
          }
      }

    | expr '-' expr
      {
          char buf[100];

          /* Constant Folding */
          if (isNumber($1) && isNumber($3))
          {
              sprintf(buf, "%d", atoi($1) - atoi($3));
              $$ = strdup(buf);

              printf("// Constant Folding: %s - %s -> %s\n",
                     $1, $3, $$);
          }

          /* Algebraic Transformation: x - 0 -> x */
          else if (strcmp($3, "0") == 0)
          {
              $$ = strdup($1);

              printf("// Algebraic Simplification: x - 0 -> x\n");
          }

          else
          {
              sprintf(buf, "%s - %s", $1, $3);
              $$ = strdup(buf);
          }
      }

    | expr '*' expr
      {
          char buf[100];

          /* Constant Folding */
          if (isNumber($1) && isNumber($3))
          {
              sprintf(buf, "%d", atoi($1) * atoi($3));
              $$ = strdup(buf);

              printf("// Constant Folding: %s * %s -> %s\n",
                     $1, $3, $$);
          }

          /* Algebraic Transformation: x * 1 -> x */
          else if (strcmp($3, "1") == 0)
          {
              $$ = strdup($1);

              printf("// Algebraic Simplification: x * 1 -> x\n");
          }

          /* Strength Reduction: x * 2 -> x + x */
          else if (strcmp($3, "2") == 0)
          {
              sprintf(buf, "%s + %s", $1, $1);
              $$ = strdup(buf);

              printf("// Strength Reduction: x * 2 -> x + x\n");
          }

          /* Algebraic Transformation: 1 * x -> x */
          else if (strcmp($1, "1") == 0)
          {
              $$ = strdup($3);

              printf("// Algebraic Simplification: 1 * x -> x\n");
          }

          else
          {
              sprintf(buf, "%s * %s", $1, $3);
              $$ = strdup(buf);
          }
      }

    | expr '/' expr
      {
          char buf[100];

          /* Constant Folding */
          if (isNumber($1) && isNumber($3) &&
              atoi($3) != 0)
          {
              sprintf(buf, "%d", atoi($1) / atoi($3));
              $$ = strdup(buf);

              printf("// Constant Folding: %s / %s -> %s\n",
                     $1, $3, $$);
          }

          /* Algebraic Transformation: x / 1 -> x */
          else if (strcmp($3, "1") == 0)
          {
              $$ = strdup($1);

              printf("// Algebraic Simplification: x / 1 -> x\n");
          }

          else
          {
              sprintf(buf, "%s / %s", $1, $3);
              $$ = strdup(buf);
          }
      }

    | '(' expr ')'
      {
          $$ = $2;
      }
    ;

%%

int main()
{
    printf("Enter Three Address Code statements (end with Ctrl+D):\n");

    yyparse();

    return 0;
}

int yyerror(char *s)
{
    printf("Syntax Error: %s\n", s);

    return 0;
}
''')


# ============================================================
# Remove old generated files
# ============================================================

!rm -f optimize.tab.c optimize.tab.h lex.yy.c optimize


# ============================================================
# Generate BISON and FLEX files
# ============================================================

!bison -d optimize.y
!flex optimize.l


# ============================================================
# Compile
# ============================================================

!gcc lex.yy.c optimize.tab.c -o optimize -lfl


# ============================================================
# Sample Input
# ============================================================

with open("input.txt", "w") as f:
    f.write("""a = 2 + 4;
b = d * 1;
c = s * 2;
""")


# ============================================================
# Execute
# ============================================================

import subprocess

result = subprocess.run(
    ["./optimize"],
    stdin=open("input.txt", "r"),
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

Enter Three Address Code statements (end with Ctrl+D):
// Constant Folding: 2 + 4 -> 6
a = 6
// Algebraic Simplification: x * 1 -> x
b = d
// Strength Reduction: x * 2 -> x + x
c = s + s



## RESULT
Thus, the FLEX and BISON program for simple code optimization techniques — constant folding, strength reduction, and algebraic simplification — was successfully implemented and tested with various inputs.
